# Task 1: Evaluate XGBoost Model — Collect Metrics for Technical Writeup

Loads saved models (or trains from scratch if none cached), then runs inference on
train and test data to produce all metrics needed for the writeup:

- **Train**: MSE and R² per horizon
- **Test**: min, max, mean speed per horizon
- **Submission**: write prediction CSV and validate


In [ ]:
import csv, json, pickle
from pathlib import Path
from datetime import datetime
import numpy as np
import xgboost as xgb

DATA = Path("../../dataset-task1")
WINDOW = 15
HORIZONS = [5, 10, 15]
HORIZON_NAMES = ["h5 (20 min)", "h10 (40 min)", "h15 (60 min)"]
N_ROADS = 1260

MODEL_DIR = Path("../../submissions") / "xgb_models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)


## 1. Data Loading

In [ ]:
def load_train_speeds():
    s1 = np.load(DATA / "train" / "train_speed_m1_1_11160.npy")
    s2 = np.load(DATA / "train" / "train_speed_m2_1_5039.npy")
    return s1, s2

def load_test_data():
    hist = np.load(DATA / "test" / "test_X_hist.npy")
    with open(DATA / "test" / "test_texts.json") as f:
        texts_dict = json.load(f)
    keys = sorted(texts_dict.keys(), key=lambda k: int(k.split("_")[1]))
    return hist, [texts_dict[k] for k in keys]

def load_adjacency():
    return np.load(DATA / "static" / "matrix.npy")

def load_roads():
    with open(DATA / "static" / "Roads1260.json") as f:
        return json.load(f)

s1, s2 = load_train_speeds()
adj = load_adjacency()
roads = load_roads()

print(f"Block1: {s1.shape}, Block2: {s2.shape}")
print(f"Adjacency: {adj.shape}, nnz={np.count_nonzero(adj)}")


## 2. Build Supervised Windows

In [ ]:
def build_windows(speeds, stride=1):
    T = len(speeds)
    max_horizon = max(HORIZONS)
    max_t = T - WINDOW - max_horizon
    X, Y = [], []
    for t in range(0, max_t, stride):
        X.append(speeds[t : t + WINDOW])
        y = np.stack([speeds[t + WINDOW + h] for h in HORIZONS], axis=0)
        Y.append(y)
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)

X1, Y1 = build_windows(s1)
X2, Y2 = build_windows(s2)
X_all = np.concatenate([X1, X2], axis=0).astype(np.float32)
Y_all = np.concatenate([Y1, Y2], axis=0).astype(np.float32)
print(f"Train windows: {len(X_all)}, X: {X_all.shape}, Y: {Y_all.shape}")


## 3. Feature Engineering (same as 05)

In [ ]:
def build_features(hist_windows, adj, roads):
    N, T, R = hist_windows.shape

    roadclass = np.array([roads[r][0].get("roadclass", 0) for r in range(R)], dtype=np.float32)
    length = np.array([roads[r][0].get("length", 0) for r in range(R)], dtype=np.float32)

    lags = np.stack(
        [hist_windows[:, -1, :],
         hist_windows[:, -2, :],
         hist_windows[:, -4, :],
         hist_windows[:, -8, :],
         hist_windows[:,  0, :]],
        axis=-1,
    )

    mean_h = hist_windows.mean(axis=1)
    std_h = hist_windows.std(axis=1)
    trend = hist_windows[:, -1, :] - hist_windows[:, 0, :]

    degrees = adj.sum(axis=1, keepdims=True).clip(min=1)
    adj_norm = adj.astype(np.float32) / degrees.astype(np.float32)

    neighbor_last = hist_windows[:, -1, :] @ adj_norm.T
    neighbor_3 = hist_windows[:, -3, :] @ adj_norm.T

    feats = np.stack(
        [lags[:, :, 0], lags[:, :, 1], lags[:, :, 2], lags[:, :, 3], lags[:, :, 4],
         mean_h, std_h, trend,
         np.broadcast_to(roadclass, (N, R)),
         np.broadcast_to(length, (N, R)),
         neighbor_last, neighbor_3],
        axis=-1,
    )
    return feats.reshape(-1, 12).astype(np.float32)

F_all = build_features(X_all, adj, roads)
Y_flat = Y_all.transpose(0, 2, 1).reshape(-1, 3)
print(f"Train features: {F_all.shape}, Targets: {Y_flat.shape}")


## 4. Load or Train Models

In [ ]:
models = {}
model_paths = {}
for hi, hn in enumerate(["h5", "h10", "h15"]):
    p = MODEL_DIR / f"xgb_{hn}.pkl"
    model_paths[hn] = p
    if p.exists():
        with open(p, "rb") as f:
            models[hn] = pickle.load(f)
        print(f"  Loaded {hn} from {p}")
    else:
        model = xgb.XGBRegressor(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            subsample=0.8, verbosity=0, n_jobs=-1,
        )
        model.fit(F_all, Y_flat[:, hi])
        models[hn] = model
        with open(p, "wb") as f:
            pickle.dump(model, f)
        print(f"  Trained and saved {hn} to {p}")


## 5. Evaluate on Training Data

Compute MSE and R² per horizon. These go in the "Training Performance" table
in the technical writeup.


In [ ]:
print("\n=== TRAINING PERFORMANCE ===\n")
train_metrics = {}
for hi, (hn, hname) in enumerate(zip(["h5", "h10", "h15"], HORIZON_NAMES)):
    y_true = Y_flat[:, hi]
    y_pred = models[hn].predict(F_all)

    mse = np.mean((y_true - y_pred) ** 2)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot
    rmse = np.sqrt(mse)

    train_metrics[hn] = {"mse": mse, "rmse": rmse, "r2": r2}
    print(f"  {hname}:")
    print(f"    MSE:  {mse:.6f}")
    print(f"    RMSE: {rmse:.4f}")
    print(f"    R²:   {r2:.6f}")

print(f"\n  Overall MSE: {np.mean([m['mse'] for m in train_metrics.values()]):.6f}")


## 6. Inference on Test Set

In [ ]:
test_hist, _ = load_test_data()
F_test = build_features(test_hist, adj, roads)
print(f"Test features: {F_test.shape}")

test_preds = np.zeros((540, 3, N_ROADS), dtype=np.float32)
for hi, hn in enumerate(["h5", "h10", "h15"]):
    test_preds[:, hi, :] = models[hn].predict(F_test).reshape(540, N_ROADS)
test_preds = test_preds.clip(min=0)

print("\n=== TEST PREDICTION STATISTICS ===\n")
for hi, hname in enumerate(HORIZON_NAMES):
    pred = test_preds[:, hi, :]
    print(f"  {hname}:")
    print(f"    Min:   {pred.min():.2f}")
    print(f"    Max:   {pred.max():.2f}")
    print(f"    Mean:  {pred.mean():.2f}")
    print(f"    Std:   {pred.std():.2f}")

print(f"\n  Overall mean: {test_preds.mean():.2f}")
print(f"  Overall std:  {test_preds.std():.2f}")
print(f"  % zero values: {100 * (test_preds == 0).mean():.2f}%")


## 7. Feature Importance (for Discussion section)

In [ ]:
feature_names = [
    "lag_t0", "lag_t1", "lag_t3", "lag_t7", "lag_earliest",
    "mean_15", "std_15", "trend",
    "roadclass", "length",
    "neighbor_last", "neighbor_3",
]

print("\n=== FEATURE IMPORTANCE ===\n")
for hn, hname in zip(["h5", "h10", "h15"], HORIZON_NAMES):
    fi = models[hn].feature_importances_
    sorted_idx = np.argsort(fi)[::-1]
    print(f"  {hname}:")
    for i in sorted_idx:
        print(f"    {feature_names[i]:<18s} {fi[i]:.4f}")
    print()


## 8. Write Submission CSV (optional)

In [ ]:
def write_submission(predictions, label="xgb_eval"):
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_dir = Path("../../submissions") / f"{ts}_{label}"
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / "submission.csv"
    with open(out_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["id", "speed"])
        for s in range(540):
            for hi, hn in enumerate(["h5", "h10", "h15"]):
                for r in range(N_ROADS):
                    writer.writerow([f"test_{s:05d}_{hn}_r{r}", f"{predictions[s, hi, r]:.6f}"])

    expected = 540 * 3 * N_ROADS + 1
    with open(out_path) as f:
        actual = sum(1 for _ in f)
    assert actual == expected, f"Row count: {actual} vs {expected}"
    print(f"Saved {out_path} ({actual - 1} rows)")

    return out_dir

out_dir = write_submission(test_preds)


## 9. Summary — Copy-Ready for Writeup

Run this cell and copy each value into the corresponding table in the technical writeup.


In [ ]:
print("=" * 60)
print("COPY THESE VALUES INTO task1_technical_report.md")
print("=" * 60)

print("\n--- Training Performance ---")
for hn, hname in zip(["h5", "h10", "h15"], HORIZON_NAMES):
    m = train_metrics[hn]
    print(f"  {hname}: MSE={m['mse']:.6f} | R²={m['r2']:.6f}")

print("\n--- Test Prediction Statistics ---")
for hi, hname in enumerate(HORIZON_NAMES):
    pred = test_preds[:, hi, :]
    print(f"  {hname}: Min={pred.min():.2f} | Max={pred.max():.2f} | Mean={pred.mean():.2f}")

print(f"\n--- Overall ---")
print(f"  Overall train MSE: {np.mean([m['mse'] for m in train_metrics.values()]):.6f}")
print(f"  Test mean speed:   {test_preds.mean():.2f}")
print(f"  Zero predictions:  {100 * (test_preds == 0).mean():.2f}%")

print("\n" + "=" * 60)
